## Setup

In [0]:
pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
mlflow.autolog()

## Get the conf from the local conf file
model_config = mlflow.models.ModelConfig(development_config="../conf/chapter05_conf.yml")

retriever_configs = model_config.get("retriever_configs")

## Single Retriever

In [0]:
from typing import List, Dict, Optional, Any
from databricks.vector_search.client import VectorSearchClient

from mlflow.entities import SpanType, Document


class VectorSearchWrapper:
    def __init__(self, retriever_config: Dict):
        """
        Initialize the VectorSearchWrapper with a single retriever config.

        :param retriever_config: Dictionary containing the retriever config.
            Example:
            {
              'endpoint_name': 'vs_endpoint',
              'index_name': 'workspace.unity_air.faq_index',
              'columns': ['id', 'question', 'answer', 'search_text'],
              'k': 3,
              'retriever_schema': {
                  'primary_key': 'id',
                  'text_column': 'search_text',
                  'doc_uri': 'id',
                  'name': 'unity_air_faq_vs_index'
              }
            }
        """
        self.vsc = VectorSearchClient()
        self.retriever_cfg = retriever_config

        # Create the index handle
        self.index = self.vsc.get_index(
            endpoint_name=retriever_config["endpoint_name"],
            index_name=retriever_config["index_name"],
        )
    
    @mlflow.trace(span_type=SpanType.RETRIEVER, name="single_retriever_search", attributes={"vs_type": "databricks_vector_search"})
    def search(
        self,
        query_text: str,
        columns: Optional[List[str]] = None,
        filters: Optional[Dict[str, tuple]] = None,
        num_results: Optional[int] = None,
    ):
        """
        Perform a similarity search against the retriever's index.

        :param query_text: The text query to search for.
        :param columns: Optional override for columns to return.
        :param filters: Optional filters to apply.
        :param num_results: Optional override for number of results (k).
        :return: Search results from the vector index.
        """
        
        mlflow.update_current_trace(tags={"vs_endpoint_name": self.retriever_cfg["endpoint_name"]})
        mlflow.update_current_trace(tags={"vs_index_name": self.retriever_cfg["index_name"]})

        span = mlflow.get_current_active_span()
        span.set_attribute("filters", filters or self.retriever_cfg.get("filters", {}))
        span.set_attribute("retriever_k", num_results or self.retriever_cfg.get("k", 5))
                             
        return self.index.similarity_search(
            query_text=query_text,
            columns=columns or self.retriever_cfg.get("columns", []),
            filters=filters or self.retriever_cfg.get("filters", {}),
            num_results=num_results or self.retriever_cfg.get("k", 5),
        )
    
    @mlflow.trace(span_type=SpanType.PARSER, name="parse_search_result")
    def parse_search_result(self, search_result):
        columns = search_result['manifest']["columns"]
        data_array = search_result.get("result").get("data_array")

        retriever_schema = self.retriever_cfg['retriever_schema']

        mapped_result = {}
        output_list = []

        if len(data_array) > 0:
            for data in data_array:
                for column, column_value in zip(columns, data):
                    mapped_result[column['name']] = column_value
                
                metadata = {'score': mapped_result['score']}
                doc = Document(
                    page_content = mapped_result[retriever_schema['text_column']],
                    metadata = metadata,
                    id = mapped_result[retriever_schema['primary_key']]
                )
                output_list.append(doc)

        return output_list
    
    @mlflow.trace(span_type=SpanType.RETRIEVER)
    def refined_search(
        self,
        query_text: str,
        columns: Optional[List[str]] = None,
        filters: Optional[Dict[str, tuple]] = None,
        num_results: Optional[int] = None,
        ):

        search_results = self.search(
            query_text=query_text,
            columns=columns,
            filters=filters,
            num_results=num_results,
        )

        parsed_results = self.parse_search_result(search_result=search_results)

        return parsed_results

In [0]:
client = VectorSearchWrapper(retriever_configs['retriever_1'])

# Normal search (docs don't render)
results = client.search("hello")

In [0]:
# Refined search (docs render)
refined_search = client.refined_search("hello")

## Multi Retriever

In [0]:
from typing import List, Dict, Optional, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
import contextvars
import mlflow
from mlflow.entities import SpanType

mlflow.langchain.autolog()

class MultiRetrieverOrchestrator:
    def __init__(
        self,
        retriever_configs: List[Dict[str, Any]],
        llm_endpoint: str = "databricks-gpt-oss-120b",
    ):
        self.retrievers = [VectorSearchWrapper(retriever_configs[config]) for config in retriever_configs]
        self.llm = ChatDatabricks(endpoint=llm_endpoint, temperature=0)

    def generate_retriever_queries(self, query_text: str) -> List[str]:
        """
        Use the LLM to generate a list of queries, one per retriever.
        """
        num_queries = len(self.retrievers)
        response = self.llm.invoke(
            f"Generate {num_queries} variations of this query for vector search: '{query_text}'. "
            f"Return the generated queries in a list. For example ['how are you?', 'how do you do?']"
        )

        text_items = [
            item["text"] for item in response.content if item.get("type") == "text"
        ]
        if not text_items:
            raise ValueError("LLM response does not contain any items with type='text'")

        import ast

        try:
            queries = ast.literal_eval(text_items[0])
        except Exception as e:
            raise ValueError(f"Failed to parse LLM text as list: {e}")

        if len(queries) != num_queries:
            raise ValueError(
                f"LLM returned {len(queries)} queries, expected {num_queries}"
            )

        return queries

    @mlflow.trace(span_type=SpanType.RETRIEVER)
    def execute_parallel_search(
        self,
        queries: List[str],
        columns: Optional[List[str]] = None,
        filters: Optional[Dict[str, tuple]] = None,
        num_results: Optional[int] = None,
        refined: bool = True,
    ):
        """
        Execute retrievers in parallel using provided queries.
        """
        combined_results: List[Any] = []

        with ThreadPoolExecutor(max_workers=len(self.retrievers)) as executor:
            futures = {}
            for retriever, query in zip(self.retrievers, queries):
                ctx = contextvars.copy_context()
                search_fn = retriever.refined_search if refined else retriever.search
                futures[
                    executor.submit(
                        ctx.run, search_fn, query, columns, filters, num_results
                    )
                ] = retriever

            for future in as_completed(futures):
                try:
                    retriever_results = future.result()
                    if isinstance(retriever_results, list):
                        combined_results.extend(retriever_results)
                except Exception as e:
                    print(f"Error in retriever {futures[future]}: {e}")

        return combined_results

    def query_rewriting_search(self, query_text: str):
        with mlflow.start_span(span_type=SpanType.CHAIN) as span:
            span.set_inputs({"query": query_text})

            llm_queries = self.generate_retriever_queries(query_text)
            search_results = self.execute_parallel_search(llm_queries)

            span.set_outputs(search_results)

            return search_results


In [0]:
orchestrator = MultiRetrieverOrchestrator(retriever_configs)

In [0]:
orchestrator.query_rewriting_search(query_text="hello")

## TODO List
**Completed**
- Customize Span with Decorator (Done)
- Customize Span with Context Manager(Done)
- Trace Tags (Done)
- Trace attributes (Done)
- Combine Auto Trace (Done)
- Multi Threading (Done)
- Retriever schema (Done)

**Pending**
- LangChain Custom Callback (do with a reranker)
- Streaming (Complete the RAG chain (?))
- Asynchronous Tracing
- Search Trace
- Low Level API (?)